# Strategy: planificacion de sesiones de estudio

## Introduccion

Una aplicacion de estudio puede recomendar una sesion intensiva o una sesion distribuida. Ambas reciben el mismo numero de temas y minutos disponibles, pero calculan bloques diferentes. Strategy encapsula estas reglas y permite cambiar el plan en tiempo de ejecucion.

## Sin patron

El planificador concentra todas las reglas en condicionales. Agregar una modalidad nueva obliga a editar el mismo metodo y aumenta su complejidad.

In [3]:
# Las reglas estan mezcladas: cada modalidad agrega otra rama.
def crear_plan(modalidad, temas, minutos):
    if modalidad == "intensivo":
        return [minutos // len(temas)] * len(temas)
    if modalidad == "distribuido":
        pausa = 10 * (len(temas) - 1)
        return [(minutos - pausa) // len(temas)] * len(temas)
    raise ValueError("Modalidad desconocida")

temas = ["POO", "SOLID", "Patrones"]
print(crear_plan("intensivo", temas, 90))
print(crear_plan("distribuido", temas, 90))

[30, 30, 30]
[23, 23, 23]


## Con Strategy

Roles: contexto (`PlanificadorEstudio`), interfaz (`EstrategiaEstudio`) y dos estrategias concretas (`PlanIntensivo`, `PlanDistribuido`). El algoritmo puede cambiarse sin modificar el contexto.

In [4]:
from abc import ABC, abstractmethod

# Strategy: contrato comun para todos los algoritmos de planificacion.
class EstrategiaEstudio(ABC):
    @abstractmethod
    def crear_plan(self, temas, minutos):
        raise NotImplementedError

# Estrategia concreta que elimina pausas y reparte todo el tiempo.
class PlanIntensivo(EstrategiaEstudio):
    def crear_plan(self, temas, minutos):
        return [minutos // len(temas)] * len(temas)

# Estrategia concreta que reserva diez minutos entre temas.
class PlanDistribuido(EstrategiaEstudio):
    def crear_plan(self, temas, minutos):
        pausa = 10 * (len(temas) - 1)
        bloque = (minutos - pausa) // len(temas)
        return [bloque] * len(temas)

# Contexto: coordina el algoritmo sin conocer sus formulas internas.
class PlanificadorEstudio:
    def __init__(self, estrategia: EstrategiaEstudio):
        self.estrategia = estrategia

    def cambiar_estrategia(self, estrategia: EstrategiaEstudio):
        # El algoritmo puede cambiar durante la ejecucion.
        self.estrategia = estrategia

    def planificar(self, temas, minutos):
        # Delegacion polimorfica: cada estrategia calcula su propio plan.
        return self.estrategia.crear_plan(temas, minutos)

temas = ["POO", "SOLID", "Patrones"]
planificador = PlanificadorEstudio(PlanIntensivo())
print("Intensivo:", planificador.planificar(temas, 90))
planificador.cambiar_estrategia(PlanDistribuido())
print("Distribuido:", planificador.planificar(temas, 90))

Intensivo: [30, 30, 30]
Distribuido: [23, 23, 23]


## UML

```plantuml
@startuml
interface EstrategiaEstudio {
  +crear_plan(temas, minutos)
}
class PlanIntensivo
class PlanDistribuido
class PlanificadorEstudio {
  -estrategia: EstrategiaEstudio
  +cambiar_estrategia(estrategia)
  +planificar(temas, minutos)
}
EstrategiaEstudio <|.. PlanIntensivo
EstrategiaEstudio <|.. PlanDistribuido
PlanificadorEstudio o--> EstrategiaEstudio
@enduml
```

## Justificacion

Elegi Strategy porque existen varias reglas para resolver la misma tarea y el usuario puede cambiar de modalidad durante la ejecucion. Factory Method se centra en crear productos y Adapter en traducir interfaces; este problema necesita encapsular y reemplazar algoritmos.